# 001 OSM Prefecture Network Check

OpenStreetMap から都道府県単位の自動車道路ネットワークを取得し、ノード数・エッジ数・最短経路計算が可能かを検証します。

In [ ]:
from __future__ import annotations

import json
import random
import re
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import osmnx as ox

In [ ]:
PLACE_NAME = "Tokyo, Japan"
SEED = 42
NOTEBOOK_NAME = "001_osm_prefecture_network_check"


def slugify(value: str) -> str:
    normalized = value.strip().lower()
    normalized = re.sub(r"[^a-z0-9]+", "_", normalized)
    return normalized.strip("_")


place_slug = slugify(PLACE_NAME)

figures_dir = Path("outputs/figures") / NOTEBOOK_NAME / place_slug
metrics_dir = Path("outputs/metrics") / NOTEBOOK_NAME / place_slug
routes_dir = Path("outputs/routes") / NOTEBOOK_NAME / place_slug

for directory in (figures_dir, metrics_dir, routes_dir):
    directory.mkdir(parents=True, exist_ok=True)

print(f"PLACE_NAME: {PLACE_NAME}")
print(f"figures_dir: {figures_dir}")
print(f"metrics_dir: {metrics_dir}")
print(f"routes_dir: {routes_dir}")

In [ ]:
def largest_strongly_connected_subgraph(graph: nx.MultiDiGraph) -> nx.MultiDiGraph:
    components = nx.strongly_connected_components(graph)
    largest_component_nodes = max(components, key=len)
    return graph.subgraph(largest_component_nodes).copy()


def save_network_plot(graph: nx.MultiDiGraph, output_path: Path) -> None:
    fig, _ = ox.plot_graph(
        graph,
        node_size=0,
        edge_color="dimgray",
        edge_linewidth=0.4,
        bgcolor="white",
        show=False,
        close=False,
    )
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def save_route_plot(graph: nx.MultiDiGraph, route: list[int], output_path: Path) -> None:
    fig, _ = ox.plot_graph_route(
        graph,
        route,
        route_color="crimson",
        route_linewidth=3,
        node_size=0,
        edge_color="lightgray",
        edge_linewidth=0.4,
        bgcolor="white",
        show=False,
        close=False,
    )
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

In [ ]:
random.seed(SEED)

print("OpenStreetMapから道路ネットワークを取得しています...")
graph = ox.graph_from_place(PLACE_NAME, network_type="drive", simplify=True)

node_count = graph.number_of_nodes()
edge_count = graph.number_of_edges()

print("取得完了")
print(f"ノード数: {node_count:,}")
print(f"エッジ数: {edge_count:,}")

network_image_path = figures_dir / "road_network.png"
save_network_plot(graph, network_image_path)
print(f"道路ネットワーク画像を保存しました: {network_image_path}")

In [ ]:
routed_graph = largest_strongly_connected_subgraph(graph)
routed_nodes = list(routed_graph.nodes)
origin, destination = random.sample(routed_nodes, 2)

print("最短経路を計算しています...")
route = nx.shortest_path(
    routed_graph,
    source=origin,
    target=destination,
    weight="length",
)
route_length_m = nx.shortest_path_length(
    routed_graph,
    source=origin,
    target=destination,
    weight="length",
)

print(f"出発ノード: {origin}")
print(f"到着ノード: {destination}")
print(f"最短経路距離: {route_length_m:,.2f} m")

route_image_path = figures_dir / "shortest_route.png"
save_route_plot(routed_graph, route, route_image_path)
print(f"最短経路画像を保存しました: {route_image_path}")

In [ ]:
metrics = {
    "place_name": PLACE_NAME,
    "seed": SEED,
    "node_count": node_count,
    "edge_count": edge_count,
    "origin_node": origin,
    "destination_node": destination,
    "shortest_path_distance_m": route_length_m,
    "route_node_count": len(route),
}

metrics_json_path = metrics_dir / "network_metrics.json"
metrics_csv_path = metrics_dir / "network_metrics.csv"
route_nodes_path = routes_dir / "route_nodes.json"
route_geojson_path = routes_dir / "route_edges.geojson"

metrics_json_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
metrics_csv_path.write_text(
    "metric,value\n"
    + "\n".join(f"{key},{value}" for key, value in metrics.items()),
    encoding="utf-8",
)
route_nodes_path.write_text(
    json.dumps({"place_name": PLACE_NAME, "route_nodes": route}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

route_edges_gdf = ox.routing.route_to_gdf(routed_graph, route, weight="length")
route_edges_gdf.to_file(route_geojson_path, driver="GeoJSON")

print(f"メトリクスJSONを保存しました: {metrics_json_path}")
print(f"メトリクスCSVを保存しました: {metrics_csv_path}")
print(f"経路ノードJSONを保存しました: {route_nodes_path}")
print(f"経路GeoJSONを保存しました: {route_geojson_path}")

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(plt.imread(network_image_path))
plt.axis("off")
plt.title("Road Network")

plt.figure(figsize=(8, 8))
plt.imshow(plt.imread(route_image_path))
plt.axis("off")
plt.title("Shortest Route")
plt.show()